# SentinelIQ — Graph Feature Engineering

This notebook explores the NetworkX account-device-IP relationship graph and the graph-derived features used to detect synthetic identity rings and account takeover patterns.

**What we cover:**
1. Graph construction from transaction events
2. Graph structure analysis (nodes, edges, components)
3. Fraud ring detection via shared device/IP analysis
4. Graph feature extraction (degree centrality, component size, shared devices)
5. Visualising fraud rings with NetworkX and Matplotlib
6. Feature importance of graph features in the ML model

> **Prerequisites:** Run `python scripts/generate_data.py` and `python scripts/ingest_and_run.py` first.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from dotenv import load_dotenv

load_dotenv('../.env')
print('Imports complete.')

## 1. Load the Account-Device-IP Graph

In [ ]:
with open('../data/graphs/account_graph.pkl', 'rb') as f:
    G = pickle.load(f)

print(f'Graph loaded successfully.')
print(f'Nodes: {G.number_of_nodes():,}')
print(f'Edges: {G.number_of_edges():,}')

# Count node types
node_types = Counter(data.get('node_type', 'unknown') for _, data in G.nodes(data=True))
print(f'\nNode type breakdown:')
for ntype, count in sorted(node_types.items()):
    print(f'  {ntype:10s}: {count:,}')

## 2. Graph Structure Analysis

In [ ]:
# Connected components analysis
components = list(nx.connected_components(G))
component_sizes = sorted([len(c) for c in components], reverse=True)

print(f'Total connected components: {len(components):,}')
print(f'Largest component size: {component_sizes[0]:,} nodes')
print(f'Median component size: {np.median(component_sizes):.1f} nodes')
print(f'\nTop 10 component sizes: {component_sizes[:10]}')

# Distribution of component sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(component_sizes, bins=30, color='#58a6ff', edgecolor='white')
axes[0].set_xlabel('Component Size (nodes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Connected Component Sizes')
axes[0].set_yscale('log')

# Degree distribution
degrees = [d for _, d in G.degree()]
axes[1].hist(degrees, bins=30, color='#d29922', edgecolor='white')
axes[1].set_xlabel('Node Degree')
axes[1].set_ylabel('Count')
axes[1].set_title('Node Degree Distribution')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('../data/processed/graph_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Fraud Ring Detection

Synthetic identity rings are visible as device or IP nodes with high degree (connected to many accounts).

In [ ]:
# Find device nodes shared by multiple accounts (fraud ring signal)
fraud_rings = []

for node, data in G.nodes(data=True):
    if data.get('node_type') == 'device':
        neighbors = list(G.neighbors(node))
        account_neighbors = [
            n for n in neighbors
            if G.nodes[n].get('node_type') == 'account'
        ]
        if len(account_neighbors) >= 3:
            fraud_rings.append({
                'device': node,
                'accounts': account_neighbors,
                'ring_size': len(account_neighbors)
            })

fraud_rings.sort(key=lambda x: x['ring_size'], reverse=True)

print(f'Potential fraud rings detected: {len(fraud_rings)}')
print(f'\nTop 5 rings by size:')
for ring in fraud_rings[:5]:
    print(f"  Device: {ring['device']} → {ring['ring_size']} accounts")

In [ ]:
# Visualise the largest fraud ring
if fraud_rings:
    top_ring = fraud_rings[0]
    ring_nodes = [top_ring['device']] + top_ring['accounts']
    
    # Also include IP nodes connected to these accounts
    for acc in top_ring['accounts']:
        for neighbor in G.neighbors(acc):
            if G.nodes[neighbor].get('node_type') == 'ip':
                ring_nodes.append(neighbor)
    
    ring_subgraph = G.subgraph(set(ring_nodes))
    
    # Color nodes by type
    color_map = {
        'account': '#58a6ff',
        'device': '#d29922',
        'ip': '#f85149'
    }
    node_colors = [
        color_map.get(ring_subgraph.nodes[n].get('node_type', 'account'), '#8b949e')
        for n in ring_subgraph.nodes()
    ]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(ring_subgraph, seed=42, k=2)
    
    nx.draw_networkx(
        ring_subgraph, pos=pos, ax=ax,
        node_color=node_colors,
        node_size=800,
        font_size=7,
        font_color='white',
        edge_color='#30363d',
        width=2
    )
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#58a6ff', label='Account'),
        mpatches.Patch(color='#d29922', label='Device (shared hub)'),
        mpatches.Patch(color='#f85149', label='IP Address'),
    ]
    ax.legend(handles=legend_patches, loc='upper left')
    ax.set_title(f'Synthetic Identity Ring — {top_ring["ring_size"]} accounts sharing device {top_ring["device"]}')
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../data/processed/fraud_ring_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Ring visualisation saved.')

## 4. Graph Feature Extraction

In [ ]:
from src.ml.graph_features import GraphFeatureExtractor

extractor = GraphFeatureExtractor('../data/graphs/account_graph.pkl')

# Extract features for a sample of accounts
df_events = pd.read_csv('../data/synthetic/events.csv')
sample_accounts = df_events['account_id'].unique()[:20]

print('Graph features for sample accounts:')
print(f'{"Account ID":<15} {"Degree Centrality":<20} {"Component Size":<18} {"Shared Devices":<18} {"IP Reuse"}')
print('-' * 80)

features_list = []
for acc_id in sample_accounts:
    feats = extractor.extract(str(acc_id))
    feats['account_id'] = acc_id
    features_list.append(feats)
    print(f'{acc_id:<15} {feats["degree_centrality"]:<20.6f} {feats["component_size"]:<18} {feats["shared_device_count"]:<18} {feats["ip_reuse_count"]}')

df_features = pd.DataFrame(features_list)

In [ ]:
# Compare graph features between fraud and legitimate accounts
df_merged = df_events.merge(
    df_events.groupby('account_id')['is_fraud'].max().reset_index(),
    on='account_id'
).drop_duplicates('account_id')[['account_id', 'is_fraud_y']].rename(columns={'is_fraud_y': 'is_fraud'})

# Extract features for all unique accounts
all_features = []
for _, row in df_merged.iterrows():
    feats = extractor.extract(str(row['account_id']))
    feats['account_id'] = row['account_id']
    feats['is_fraud'] = row['is_fraud']
    all_features.append(feats)

df_all_feats = pd.DataFrame(all_features)

print('Mean graph features by fraud label:')
print(df_all_feats.groupby('is_fraud')[['degree_centrality', 'component_size', 'shared_device_count', 'ip_reuse_count']].mean().round(4))

In [ ]:
# Visualise graph feature distributions by class
graph_feature_cols = ['degree_centrality', 'component_size', 'shared_device_count', 'ip_reuse_count']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, col in zip(axes, graph_feature_cols):
    legit = df_all_feats[df_all_feats['is_fraud'] == 0][col]
    fraud = df_all_feats[df_all_feats['is_fraud'] == 1][col]
    
    ax.hist(legit, bins=30, alpha=0.6, color='#3fb950', label='Legitimate', density=True)
    ax.hist(fraud, bins=30, alpha=0.6, color='#f85149', label='Fraud', density=True)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Graph Feature Distributions: Fraud vs. Legitimate Accounts', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/graph_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Degree Centrality Deep Dive

Degree centrality is the most powerful graph feature — it captures how "connected" an account is to the broader fraud network.

In [ ]:
# Pre-computed centrality from the extractor
centrality = extractor._centrality

# Separate by node type
acc_centrality = {k: v for k, v in centrality.items() if k.startswith('ACC:')}
dev_centrality = {k: v for k, v in centrality.items() if k.startswith('DEV:')}
ip_centrality  = {k: v for k, v in centrality.items() if k.startswith('IP:')}

print(f'Account nodes: {len(acc_centrality):,}')
print(f'Device nodes:  {len(dev_centrality):,}')
print(f'IP nodes:      {len(ip_centrality):,}')

# Top 10 highest-centrality device nodes (likely fraud ring hubs)
top_devices = sorted(dev_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print(f'\nTop 10 highest-centrality device nodes (fraud ring hubs):')
for node, cent in top_devices:
    degree = G.degree(node)
    print(f'  {node}: centrality={cent:.6f}, degree={degree}')

In [ ]:
# Centrality distribution for account nodes
acc_cent_values = list(acc_centrality.values())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(acc_cent_values, bins=50, color='#58a6ff', edgecolor='white', log=True)
ax.set_xlabel('Degree Centrality')
ax.set_ylabel('Count (log scale)')
ax.set_title('Degree Centrality Distribution — Account Nodes')

# Mark the 95th percentile
p95 = np.percentile(acc_cent_values, 95)
ax.axvline(x=p95, color='#f85149', linestyle='--', label=f'95th percentile ({p95:.6f})')
ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/centrality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'95th percentile centrality: {p95:.6f}')
print(f'Accounts above 95th percentile: {sum(1 for v in acc_cent_values if v > p95):,}')

## 6. Graph Features in the Full Feature Matrix

In [ ]:
from src.ml.feature_engineering import build_feature_matrix

df_full = build_feature_matrix(
    events_path='../data/synthetic/events.csv',
    graph_path='../data/graphs/account_graph.pkl'
)

print('Full feature matrix with graph features:')
print(f'Shape: {df_full.shape}')
print(f'\nGraph feature columns: degree_centrality, component_size, shared_device_count, ip_reuse_count')
print(f'\nCorrelation with is_fraud:')
correlations = df_full.corr()['is_fraud'].sort_values(ascending=False)
print(correlations.to_string())

In [ ]:
# Correlation heatmap
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df_full.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5
)
ax.set_title('Feature Correlation Matrix (including graph features)')
plt.tight_layout()
plt.savefig('../data/processed/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key findings from the graph feature analysis:

- **Synthetic identity rings** are clearly visible as device nodes with high degree (3+ connected accounts)
- **`shared_device_count`** and **`component_size`** are the strongest graph-based fraud signals
- **`degree_centrality`** follows a power-law distribution — most accounts have very low centrality, but fraud ring members are outliers
- Graph features have **low correlation with tabular features**, meaning they add independent signal to the ensemble
- The combination of tabular + graph features gives the model visibility into both individual transaction anomalies AND network-level fraud patterns

This is why SentinelIQ builds a NetworkX graph: individual transactions from a synthetic identity ring look plausible in isolation, but the shared device/IP structure is unmistakable at the graph level.